# ASR Pipeline — Part 1: Normalize & Denoise

**Input:** `vietsuperspeech_500` dataset đã import sẵn trên Kaggle  
**Pipeline:** Đọc metadata → Normalize (ffmpeg loudnorm, mono 48 kHz) → Denoise (DeepFilterNet3)  
**Output:** Denoised `.wav` files + `metadata.json` cập nhật trường `denoised_path`

**Lưu ý trước khi chạy:**
- Bật **GPU** trong Kaggle (không cần Internet)
- **Cell 1** (Install) → restart kernel → **Cell 2** trở đi

In [ ]:
import sys, subprocess, os

def _run(cmd, check=True):
    print(f"\n$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip():
        print(out[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"FAILED (exit {r.returncode}): {cmd}")

# DeepFilterNet — downgrades numpy 2.x → 1.26.4 (intentional, torch 2.5.1 works fine)
_run(f"{sys.executable} -m pip install -q DeepFilterNet==0.5.6 DeepFilterLib==0.5.6")

import importlib
print("\nVerifying imports:")
for pkg, label in [("df", "DeepFilterNet")]:
    try:
        importlib.import_module(pkg); print(f"  [OK] {label}")
    except ImportError as e:
        print(f"  [FAIL] {label}: {e}")

print("\n✓ Done. Restart kernel once, then run from Cell 2 onward.")

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    print("Batch mode — skipping restart.")
else:
    os.kill(os.getpid(), 9)

In [ ]:
from pathlib import Path
import json, torch, time
from datetime import datetime

# ── Input: dataset đã import sẵn trên Kaggle ─────────────────────────────
DATASET_ROOT  = Path("/kaggle/input/datasets/nguynthikmino/vi-asr-dataset/vietsuperspeech_500")
METADATA_PATH = DATASET_ROOT / "metadata.json"

# ── Output ────────────────────────────────────────────────────────────────
WORKING_DIR  = Path("/kaggle/working")
DENOISED_DIR = WORKING_DIR / "vietsuperspeech_500_denoised"
DENOISED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_METADATA = DENOISED_DIR / "metadata.json"

# ── Denoising parameters ──────────────────────────────────────────────────
DENOISE_CHUNK_SECONDS = 30   # DeepFilterNet xử lý từng đoạn để tiết kiệm VRAM

# ── Load metadata ─────────────────────────────────────────────────────────
with open(METADATA_PATH, encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Loaded {len(metadata)} entries from {METADATA_PATH}")
print("Example entry:", json.dumps(metadata[0], ensure_ascii=False, indent=2))

print("\nCUDA:", torch.cuda.is_available(),
      f"| GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "")
print("Dataset root :", DATASET_ROOT)
print("Denoised to  :", DENOISED_DIR)

PIPELINE_T0 = time.perf_counter()
PIPELINE_STARTED_AT = datetime.now()
print("Started at:", PIPELINE_STARTED_AT.strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
# ── Normalize + DeepFilterNet ─────────────────────────────────────────────
import math, subprocess
import torch, torchaudio
from df.enhance import enhance, init_df


def normalize_audio(input_path: str, output_wav: str):
    subprocess.run(
        [
            "ffmpeg", "-y", "-i", input_path,
            "-ac", "1", "-ar", "48000",
            "-af", "loudnorm", "-c:a", "pcm_s16le", output_wav,
        ],
        check=True, capture_output=True,
    )


class NoiseReducer:
    def __init__(self, chunk_seconds: int = 30):
        print("Loading DeepFilterNet3...")
        self.model, self.df_state, _ = init_df()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.chunk_seconds = chunk_seconds
        print(f"DeepFilterNet ready on {self.device}")

    def process(self, input_wav: str, output_wav: str):
        audio, sr = torchaudio.load(input_wav)
        cs    = sr * self.chunk_seconds
        total = audio.shape[1]
        chunks = []
        for i in range(math.ceil(total / cs)):
            s, e  = i * cs, min((i + 1) * cs, total)
            chunk = audio[:, s:e].to(self.device)
            with torch.no_grad():
                chunks.append(enhance(self.model, self.df_state, chunk).cpu())
            del chunk
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        enhanced = torch.cat(chunks, dim=1)
        torchaudio.save(output_wav, enhanced, sr)


print("Normalize + NoiseReducer ready.")

In [ ]:
# ── Chạy normalize → denoise trên toàn bộ dataset ────────────────────────
import json, tempfile
from pathlib import Path
from tqdm import tqdm

noise_reducer = NoiseReducer(chunk_seconds=DENOISE_CHUNK_SECONDS)

output_metadata = []

for entry in tqdm(metadata, desc="Denoise"):
    audio_path   = DATASET_ROOT / entry["audio"]
    entry_id     = entry.get("id", Path(entry["audio"]).stem)
    denoised_wav = DENOISED_DIR / f"{entry_id}_denoised.wav"

    new_entry = {
        "id":       entry_id,
        "filename": denoised_wav.name,
        "text":     entry.get("text", ""),
    }

    if denoised_wav.exists():
        output_metadata.append(new_entry)
        continue

    with tempfile.NamedTemporaryFile(suffix=".wav", dir=WORKING_DIR, delete=False) as tmp:
        norm_wav = tmp.name

    try:
        normalize_audio(str(audio_path), norm_wav)
        noise_reducer.process(norm_wav, str(denoised_wav))
    finally:
        Path(norm_wav).unlink(missing_ok=True)

    output_metadata.append(new_entry)

# Lưu metadata (chỉ gồm id, filename, text)
with open(OUTPUT_METADATA, "w", encoding="utf-8") as f:
    json.dump(output_metadata, f, ensure_ascii=False, indent=2)

print(f"\nDone. Denoised files: {DENOISED_DIR}")
print(f"Output metadata    : {OUTPUT_METADATA}")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────
import time
from datetime import datetime

elapsed = time.perf_counter() - PIPELINE_T0
h, rem  = divmod(int(elapsed), 3600)
m, s    = divmod(rem, 60)

denoised_files = list(DENOISED_DIR.glob("*.wav"))

print(f"Total runtime  : {h:02d}:{m:02d}:{s:02d}")
print(f"Started        : {PIPELINE_STARTED_AT.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished       : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Denoised files : {len(denoised_files)} / {len(metadata)}")
print(f"Output dir     : {DENOISED_DIR}")
print(f"Metadata       : {OUTPUT_METADATA}")